# Retentioneering E-Commerce Demo

Six months of clickstream data from a consumer-electronics store (Jan-Jun 2024):
600 users, ~3 600 sessions, ~31 000 events across 17 event types.

| # | Story | Signal |
|---|---|---|
| 1 | Viral article | purchase rate drops 3x during a traffic spike |
| 2 | Checkout bug | mobile users loop between shipping_details and add_to_cart |
| 3 | Payment incident | conversion collapses; support_chat spikes |
| 4 | New vs loyal | new users browse 2x more; loyal users convert 2x better |
| 5 | Cohort drift | Jan cohort (loyal) vs Apr cohort (still exploring) |


## Setup


In [ ]:
import retentioneering as rete

df = rete.datasets.load_ecom(as_dataframe=True)

stream = rete.Eventstream(
    df,
    {
        "path_cols": ["user_id", "session_id"],
        "segment_cols": [
            "platform",
            "acquisition_channel",
            "user_cohort",
            "user_lifecycle",
        ],
    },
)

print(f"Events:   {len(df):,}")
print(f'Users:    {df["user_id"].nunique():,}')
print(f'Sessions: {df["session_id"].nunique():,}')
print(f'Date range: {df["timestamp"].min().date()} - {df["timestamp"].max().date()}')
print(f'Event types: {sorted(df["event"].unique())}')

## Overview - Transition Graph

The overall graph gives a bird's-eye view of all user flows.
Edge thickness = transition frequency. Hover an edge to see the probability.
Self-loops (product_view->product_view, search->search) are clearly visible
as thick edges pointing to themselves - users browsing images or refining queries.


In [2]:
stream.transition_graph(height=600)

## Overview - Purchase Funnel (Step Sankey)

How many sessions pass through each stage of the purchase funnel,
and where do they drop off? The funnel from add_to_cart shows clearly
that cart -> shipping_details is the biggest drop-off point.


In [3]:
stream.step_sankey(
    path_pattern="add_to_cart->.*->path_end",
    step_window=4,
    height=540,
)

## Story 1 - The Viral Article

**Context:** On Feb 29 - Mar 3, a tech blogger published a review that sent a surge
of organic traffic to the store.

**What we see in the diff graph (anomaly vs normal):**
- search->search self-loop is bright **red**: visitors are refining search queries
  repeatedly - they are discovering the store for the first time
- product_view->add_to_cart is **blue**: despite heavy browsing, they rarely buy
- support_chat is **red**: confused new visitors need help navigating

**Conclusion:** classic viral-discovery spike - high traffic, low intent.
The step sankey confirms: anomaly-period sessions reach add_to_cart far less often.


In [4]:
import pandas as pd

# The viral article ran Feb 29 - Mar 3 2024 (days 59-62 of the dataset).
# We know these dates from the analytics alert; derive the segment from timestamps.
stream_v1 = stream.add_segment(
    name="is_anomaly_period",
    func=lambda df: df["timestamp"].apply(
        lambda ts: "yes"
        if pd.Timestamp("2024-02-29") <= ts <= pd.Timestamp("2024-03-03 23:59:59")
        else "no"
    ),
)

stream_v1.transition_graph(
    diff=["is_anomaly_period", "yes", "no"],
    height=600,
)

In [5]:
stream_v1.step_sankey(
    path_pattern="add_to_cart->.*->path_end",
    diff=["is_anomaly_period", "yes", "no"],
    step_window=4,
    height=540,
)

## Story 2 - The Checkout Bug

**Context:** Apr 9 - May 9, a frontend deploy introduced a bug on mobile devices:
the back-button on the Shipping Details page sent users to add_to_cart
instead of payment_details. A diagnostic `checkout_bug` event was logged
each time this happened.

**How to find it:** Filter to sessions containing `checkout_bug` events,
then derive a segment using `add_segment`. The transition graph reveals
the loop: shipping_details -> checkout_bug -> add_to_cart -> cart ->
shipping_details -> checkout_bug -> ... The diff graph (mobile bug vs normal)
shows this loop is exclusive to the bug period.


In [6]:
# Step 1: identify sessions that hit the checkout bug
bug_sessions = set(stream.df[stream.df["event"] == "checkout_bug"]["session_id"])
print(f"Sessions that hit checkout_bug: {len(bug_sessions)}")

# Step 2: derive had_checkout_bug segment
stream_v2 = stream.add_segment(
    name="had_checkout_bug",
    func=lambda df: df["session_id"].isin(bug_sessions).map({True: "yes", False: "no"}),
)

Sessions that hit checkout_bug: 32


In [7]:
# Step 3: visualise the loop in bug-period mobile sessions
bug_mobile = stream_v2.filter_events(
    {"column": "had_checkout_bug", "values": ["yes"]}
).filter_events({"column": "platform", "values": ["mobile"]})
bug_mobile.transition_graph(height=600)

In [8]:
# Step 4: diff - mobile bug period vs mobile normal period
mobile = stream_v2.filter_events({"column": "platform", "values": ["mobile"]})
mobile.transition_graph(
    diff=["had_checkout_bug", "yes", "no"],
    height=600,
)

In [9]:
mobile.step_sankey(
    path_pattern="cart->.*->path_end",
    diff=["had_checkout_bug", "yes", "no"],
    step_window=3,
    height=540,
)

## Story 3 - Root Cause Analysis: Payment Drop

**Context:** May 19 - Jun 7, the payment gateway showed a spurious validation
error to ~70% of users at the payment_details step. A `payment_error` event
was logged each time. Most users then went to support_chat; fewer than half
retried and completed the purchase.

**What we see:**
- Step Sankey (cart->path_end, payment issue diff): payment_details->purchase
  is deep **blue** - conversion collapsed during the incident
- Transition graph: payment_error is a new bright-red node unique to this period;
  payment_details->support_chat is red; payment_details->purchase is blue
- The root cause is clear: users hit the error, seek help, and mostly abandon


In [10]:
# Step 1: identify sessions with payment_error events
pay_sessions = set(stream.df[stream.df["event"] == "payment_error"]["session_id"])
print(f"Sessions that hit payment_error: {len(pay_sessions)}")

stream_v3 = stream.add_segment(
    name="had_payment_issue",
    func=lambda df: df["session_id"].isin(pay_sessions).map({True: "yes", False: "no"}),
)

Sessions that hit payment_error: 95


In [11]:
stream_v3.step_sankey(
    path_pattern="cart->.*->path_end",
    diff=["had_payment_issue", "yes", "no"],
    step_window=3,
    height=540,
)

In [12]:
stream_v3.transition_graph(
    diff=["had_payment_issue", "yes", "no"],
    height=600,
)

## Story 4 - New vs Loyal Users

**Context:** `user_lifecycle` captures where a user is in their journey:
`new` (<30 days since registration), `returning` (30-90 days), `loyal` (>90 days).

**What we see:**
- Transition graph (new vs loyal): catalog/search/filter exploration is **red**
  (heavier for new users); product_view->add_to_cart and direct funnel edges
  are **blue** (loyal users skip the browse phase)
- Step Sankey confirms: new users take 4-5 browse steps before entering
  the funnel; loyal users convert in 1-2 steps


In [13]:
stream.transition_graph(
    diff=["user_lifecycle", "new", "loyal"],
    height=600,
)

In [14]:
stream.step_sankey(
    diff=["user_lifecycle", "new", "loyal"],
    step_window=3,
    height=540,
)

## Story 5 - Cohort Analysis

**Context:** `user_cohort` is the registration month (2024-01 through 2024-04).
January users are now 3-6 months old - they behave like loyal users.
April users registered recently - still in exploration mode.

**Individual sankeys:** Jan cohort shows a compact funnel; Apr cohort shows
a long browse phase. The diff graph highlights exactly where the cohorts diverge.


In [15]:
jan = stream.filter_events({"column": "user_cohort", "values": ["2024-01"]})
jan.step_sankey(
    step_window=3,
    height=500,
)

In [16]:
apr = stream.filter_events({"column": "user_cohort", "values": ["2024-04"]})
apr.step_sankey(
    step_window=3,
    height=500,
)

In [17]:
stream.transition_graph(
    diff=["user_cohort", "2024-01", "2024-04"],
    height=600,
)

## Platform & Channel Segmentation

**Mobile vs Desktop:** mobile users abandon checkout more (smaller screen,
less trust in entering payment details). Look for blue cart->shipping_details
and shipping_details->payment_details edges.

**Paid Search vs Organic:** paid-search visitors arrive with purchase intent;
they show stronger product_view->add_to_cart (blue = more for organic).
Organic visitors show more search/catalog exploration (red).


In [18]:
stream.transition_graph(
    diff=["platform", "mobile", "desktop"],
    height=600,
)

In [19]:
stream.transition_graph(
    diff=["acquisition_channel", "paid_search", "organic"],
    height=600,
)

## Headless Analysis - Matrices

All widgets have headless equivalents that return DataFrames with styling
applied to highlight the most important cells.


In [20]:
# Markov transition probabilities - highlight the top routes
tm = stream.transition_graph_data(values="proba_out")
(
    tm.round(3)
    .style.background_gradient(cmap="Blues", axis=None, vmin=0, vmax=0.6)
    .format("{:.3f}")
    .set_caption("Markov transition probabilities (proba_out)")
)

next_event,path_start,account_page,add_to_cart,cart,catalog,checkout_bug,compare,error_page,filter_results,home,payment_details,payment_error,product_view,promo_page,purchase,review_page,search,shipping_details,support_chat,wishlist_add,path_end
event,,,,,,,,,,,,,,,,,,,,,
path_start,0.000,0.012,0.003,0.002,0.268,0.000,0.000,0.000,0.028,0.325,0.000,0.000,0.122,0.012,0.000,0.003,0.213,0.000,0.000,0.012,0.000
account_page,0.000,0.017,0.042,0.027,0.235,0.000,0.006,0.006,0.077,0.156,0.017,0.002,0.182,0.023,0.008,0.015,0.126,0.023,0.003,0.015,0.020
add_to_cart,0.000,0.017,0.014,0.166,0.120,0.006,0.007,0.013,0.047,0.056,0.062,0.007,0.215,0.019,0.032,0.010,0.059,0.103,0.011,0.019,0.016
cart,0.000,0.016,0.132,0.028,0.101,0.007,0.012,0.015,0.035,0.061,0.098,0.009,0.143,0.028,0.060,0.011,0.055,0.138,0.017,0.020,0.014
catalog,0.000,0.021,0.033,0.019,0.190,0.001,0.008,0.008,0.107,0.109,0.007,0.001,0.269,0.024,0.004,0.010,0.127,0.013,0.008,0.025,0.018
checkout_bug,0.000,0.000,0.227,0.205,0.114,0.023,0.000,0.023,0.045,0.045,0.000,0.000,0.045,0.000,0.023,0.000,0.023,0.182,0.000,0.045,0.000
compare,0.000,0.022,0.065,0.040,0.204,0.000,0.004,0.000,0.044,0.069,0.015,0.000,0.335,0.015,0.004,0.007,0.098,0.033,0.004,0.018,0.025
error_page,0.000,0.000,0.036,0.036,0.203,0.000,0.004,0.018,0.083,0.091,0.036,0.004,0.174,0.022,0.018,0.029,0.156,0.051,0.011,0.022,0.007
filter_results,0.000,0.020,0.028,0.023,0.261,0.001,0.008,0.011,0.125,0.070,0.007,0.001,0.213,0.022,0.003,0.010,0.140,0.008,0.007,0.026,0.018


In [21]:
# Filter to purchase sessions only, then show step matrix around the funnel
buyers = stream.filter_paths(
    {
        "op": "=",
        "metric": "has",
        "value": True,
        "metric_args": {"events": "purchase"},
    }
)
print(
    f'Sessions that reached purchase: {buyers.df["session_id"].nunique():,} '
    f'({100*buyers.df["session_id"].nunique()/stream.df["session_id"].nunique():.1f}% of all)'
)

sm = buyers.step_sankey_data(
    max_steps=8,
    path_pattern="add_to_cart->.*->purchase",
)
(
    sm[0]
    .round(3)
    .style.background_gradient(cmap="Greens", axis=None, vmin=0, vmax=0.5)
    .format("{:.3f}")
    .set_caption("Step matrix: add_to_cart -> purchase (buying sessions only)")
    .highlight_max(axis=0, color="#d4f0d4")
)

Sessions that reached purchase: 2,117 (58.7% of all)


step,-8,-7,-6,-5,-4,-3,-2,-1,0,1,2,3,4,5,6,7,8
event,,,,,,,,,,,,,,,,,
path_start,0.293,0.239,0.202,0.138,0.094,0.054,0.024,0.003,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
account_page,0.010,0.024,0.024,0.017,0.030,0.024,0.024,0.020,0.000,0.017,0.034,0.010,0.013,0.013,0.024,0.020,0.007
add_to_cart,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.007,0.024,0.030,0.027,0.044,0.037,0.037
cart,0.013,0.017,0.030,0.010,0.030,0.057,0.061,0.111,0.000,0.135,0.128,0.091,0.091,0.081,0.071,0.067,0.054
catalog,0.158,0.172,0.212,0.236,0.195,0.205,0.182,0.158,0.000,0.104,0.158,0.138,0.175,0.172,0.178,0.189,0.165
checkout_bug,0.000,0.000,0.000,0.000,0.000,0.000,0.003,0.007,0.000,0.000,0.003,0.000,0.003,0.000,0.003,0.007,0.000
compare,0.003,0.010,0.007,0.003,0.027,0.017,0.017,0.013,0.000,0.010,0.007,0.003,0.000,0.003,0.010,0.003,0.007
error_page,0.003,0.007,0.010,0.010,0.017,0.010,0.007,0.000,0.000,0.017,0.000,0.013,0.013,0.007,0.003,0.013,0.013
filter_results,0.061,0.064,0.067,0.037,0.057,0.088,0.064,0.047,0.000,0.040,0.040,0.037,0.067,0.051,0.054,0.071,0.088


## Data Processors Showcase


In [22]:
# Collapse consecutive identical events (e.g. search->search->search => search)
collapsed = stream.collapse_events(repetitive=True)
print(
    f"Before: {len(stream.df):,} events  After: {len(collapsed.df):,} (self-loops collapsed)"
)

Before: 31,106 events  After: 27,053 (self-loops collapsed)


In [23]:
# Split by 30-min inactivity; use sub_session_id to avoid conflict
# with the existing session_id column
sessioned = stream.split_sessions(
    timeout=1800,
    session_col="sub_session_id",
    session_index_col="sub_session_index",
)
print(f'Original sessions:  {stream.df["session_id"].nunique():,}')
print(f'After timeout split: {sessioned.df["sub_session_id"].nunique():,}')

Original sessions:  3,605
After timeout split: 3,605


In [24]:
# Add a segment: did this session result in a purchase?
purchased = set(stream.df[stream.df["event"] == "purchase"]["session_id"])
enriched = stream.add_segment(
    name="converted",
    func=lambda df: df["session_id"].isin(purchased).map({True: "yes", False: "no"}),
)

# Diff: converting sessions vs non-converting
# Expected: converting sessions show strong funnel edges (blue = weaker for non-converters)
enriched.transition_graph(
    diff=["converted", "yes", "no"],
    height=600,
)